In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.optimizers import Adam
import seaborn as sns

In [7]:
TrainDir = "dogvscat_small/train"
TestDir = "dogvscat_small/test"
ValidDir = "dogvscat_small/validation"

In [8]:
ImgGenTrain = ImageDataGenerator(
    rescale = 1/255,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.5,
    shear_range = 0.3,
    zoom_range = 0.3,
    horizontal_flip = True,
)

TrainPP = ImgGenTrain.flow_from_directory(TrainDir, target_size=(200,200), batch_size=32, class_mode="binary")
ValidPP = ImgGenTrain.flow_from_directory(ValidDir, target_size=(200,200), batch_size=32, class_mode="binary")


Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [9]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(200, 200, 3)),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

c:\Users\hamza\OneDrive\Documents\Deep_Learning_JL\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 198, 198, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 99, 99, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 97, 97, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 46, 46, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 23, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 21, 21, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     6,554,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,795,457 (25.92 MB)

 Trainable params: 6,795,457 (25.92 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

In [12]:
model.fit(TrainPP, epochs=10, validation_data=ValidPP, batch_size=32, verbose=True)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.5000 - loss: 0.6943 - val_accuracy: 0.5000 - val_loss: 0.6922
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.5045 - loss: 0.6925 - val_accuracy: 0.5440 - val_loss: 0.6862
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.5225 - loss: 0.6921 - val_accuracy: 0.5710 - val_loss: 0.6834
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 131s 2s/step - accuracy: 0.5265 - loss: 0.6911 - val_accuracy: 0.5450 - val_loss: 0.6822
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.5165 - loss: 0.6933 - val_accuracy: 0.5220 - val_loss: 0.6870
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 119s 2s/step - accuracy: 0.5460 - loss: 0.6912 - val_accuracy: 0.5780 - val_loss: 0.6726
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 126s 2s/step - accuracy: 0.5230 - loss: 0.6902 - val_accuracy: 0.5760 - val_loss: 0.6669
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 127s 2s/step - accuracy: 0.5540 - loss: 0.6843 - val_accuracy: 0.5590 - va

In [20]:
TestDir_Img1 = "/Users/hamza/OneDrive/Documents/Deep_Learning_JL/Lesson_9/dogvscat_small/test/cats/1500.jpg"
#C:\Users\hamza\OneDrive\Documents\Deep_Learning_JL\Lesson_9\dogvscat_small\test\cats\1500.jpg

In [21]:
TestImg = load_img(TestDir_Img1, target_size=(200,200))
TestArry = img_to_array(TestImg)/255
TestArry = np.expand_dims(TestArry, axis=0)

predictions = model.predict(TestArry)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step


In [23]:
if predictions[0] < 0.5:
    print("cat")

cat


In [26]:
ValidImg = next(ValidPP)[0]
ValidLabels = next(ValidPP)[1]

In [27]:
PredictLabels = model.predict(ValidImg)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 593ms/step


In [ ]:
plt.figure(figsize=(10,12))
for i in range(0,6):
    plt.subplot(2,3,i+1)
    plt.axis("off")
    plt.imshow(TestA[i])
    plt.title(f"Actual: {"Animal" if TestArry[i]==0 else "Vehicle"}\nPredicted: {"Animal" if y_pred[i]==0 else "Vehicle"}")
plt.tight_layout()
plt.show()